<a href="https://colab.research.google.com/github/aarsha-joshi/worldwideweb/blob/main/scripts/Llama_Prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Llama Training Pair Label Generation**

In [ ]:
!pip install transformers==4.46.2 -q

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_token'))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
messages = [{"role": "user", "content": "Hello"}]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
outputs = model.generate(inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# create the directory if it doesn't exist
os.makedirs("/content/worldwideweb/data/raw", exist_ok=True)

# copy the files
!cp "/content/drive/MyDrive/World Wide Web/cleaned_data/clean_train_live.jsonl" "/content/worldwideweb/data/raw/clean_train_live.jsonl"
!cp "/content/drive/MyDrive/World Wide Web/cleaned_data/clean_train_wa.jsonl" "/content/worldwideweb/data/raw/clean_train_wa.jsonl"

print("Done!")
!ls /content/worldwideweb/data/raw/

In [ ]:
import os

os.chdir('/content')
!git clone https://token@github.com/aarsha-joshi/worldwideweb.git

# now move in
os.chdir('/content/worldwideweb')
!pip install -r requirements.txt

# verify
print(os.path.exists('data/dataset.py'))  # should print True

In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)

# restore raw data from Drive
!cp "/content/drive/MyDrive/World Wide Web/cleaned_data/clean_train_live.jsonl" "/content/worldwideweb/data/raw/clean_train_live.jsonl"
!cp "/content/drive/MyDrive/World Wide Web/cleaned_data/clean_train_wa.jsonl" "/content/worldwideweb/data/raw/clean_train_wa.jsonl"

# restore labeled progress if it exists
labeled_drive_path = "/content/drive/MyDrive/World Wide Web/cleaned_data/wm_training_pairs_labeled.jsonl"
if os.path.exists(labeled_drive_path):
    !cp "{labeled_drive_path}" "/content/worldwideweb/data/raw/wm_training_pairs_labeled.jsonl"
    print("Restored labeled checkpoint")
else:
    print("No labeled checkpoint yet, starting fresh")

# verify files are there and check sizes
!ls -lh /content/worldwideweb/data/raw/

In [ ]:
import sys
import os

# make sure we're in the right directory
os.chdir('/content/worldwideweb')

# add to path
sys.path.insert(0, '/content/worldwideweb')

# verify the data folder is there
print(os.path.exists('data/dataset.py'))  # should print True

import json
from data.dataset import NNetNavDataset
import sys
sys.path.insert(0, '/content/worldwideweb')

import json
import os
from data.dataset import NNetNavDataset


print("Loading live dataset...")
dataset_live = NNetNavDataset("data/raw/clean_train_live.jsonl")
print(f"Interactions loaded: {len(dataset_live.interactions)}")

print("Loading WA dataset...")
dataset_wa = NNetNavDataset("data/raw/clean_train_wa.jsonl")
print(f"Interactions loaded: {len(dataset_wa.interactions)}")

# combine all interactions
all_interactions = dataset_live.interactions + dataset_wa.interactions
print(f"Total interactions: {len(all_interactions)}")

# extract pairs from combined set
pairs = []
for interaction in all_interactions:
    for sc in interaction.state_changes:
        pairs.append(sc.to_wm_training_dict())

print(f"Total training pairs: {len(pairs)}")
print(f"Sample pair keys: {pairs[0].keys()}")
print(f"Sample pair: {pairs[0]}")

# filter out stop actions and unchanged states
def should_skip(pair):
    action = pair["action"].strip().lower()
    if action.startswith("stop"):
        return True
    if pair["pre_obs"] == pair["post_obs"]:
        return True
    return False

filtered_pairs = [p for p in pairs if not should_skip(p)]
print(f"After filtering stop/unchanged pairs: {len(filtered_pairs)}")

# sample down to a manageable size for Llama labeling
import random
random.seed(42)
sampled_pairs = random.sample(filtered_pairs, min(5000, len(filtered_pairs)))
print(f"Sampled {len(sampled_pairs)} pairs for labeling")

# save to jsonl
os.makedirs("data/raw", exist_ok=True)
with open("data/raw/wm_training_pairs.jsonl", "w") as f:
    for pair in sampled_pairs:
        f.write(json.dumps(pair) + "\n")

print("Saved to data/raw/wm_training_pairs.jsonl")

In [ ]:
import os

# check drive is mounted
print("=== Drive contents ===")
!ls "/content/drive/MyDrive/World Wide Web/cleaned_data/"

# check if repo data folder exists
print("\n=== Repo data/raw folder ===")
!ls /content/worldwideweb/data/raw/

In [ ]:
def get_changed_lines(pre_obs, post_obs, context_lines=5):
    """Extract only the lines that differ between pre and post obs"""
    pre_lines = set(pre_obs.splitlines())
    post_lines = set(post_obs.splitlines())

    removed = [l for l in pre_lines if l not in post_lines and l.strip()]
    added = [l for l in post_lines if l not in pre_lines and l.strip()]

    return removed, added

def call_llama(pre_obs, action, post_obs):
    removed, added = get_changed_lines(pre_obs, post_obs)

    # if diff is small enough, send just the diff
    # if no diff found, fall back to truncated full trees
    if len(removed) + len(added) == 0:
        return "No structural changes detected."

    removed_text = "\n".join(removed[:50])  # cap at 50 lines
    added_text = "\n".join(added[:50])

    prompt = f"""You are analyzing changes in a web accessibility tree.

Action taken: {action}

Elements REMOVED from the tree:
{removed_text if removed_text else "none"}

Elements ADDED to the tree:
{added_text if added_text else "none"}

Based on these changes, describe in 2-3 sentences what structurally changed
in the accessibility tree. Be specific about element types and IDs."""

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    if inputs.shape[1] > 7000:
        return "SKIPPED: input too long"

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_len = inputs.shape[1]
    new_tokens = outputs[0, prompt_len:]
    description = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return description

# test on several pairs
for i in [0, 1, 5, 10, 20, 50]:
    test = sampled_pairs[i]
    removed, added = get_changed_lines(test["pre_obs"], test["post_obs"])
    print(f"\n=== Pair {i} ===")
    print(f"Action: {test['action']}")
    print(f"DOM changed flag: {test.get('dom_changed', 'unknown')}")
    print(f"Lines removed: {len(removed)}, Lines added: {len(added)}")
    result = call_llama(test["pre_obs"], test["action"], test["post_obs"])
    print(f"Output: {result}")
    print("-" * 50)

In [ ]:
INPUT_PATH = "data/raw/wm_training_pairs.jsonl"
OUTPUT_PATH = "data/raw/wm_training_pairs_labeled.jsonl"

with open(INPUT_PATH, "r") as f:
    all_pairs = [json.loads(line) for line in f]

def get_processed_count():
    if not os.path.exists(OUTPUT_PATH):
        return 0
    with open(OUTPUT_PATH, "r") as f:
        return sum(1 for line in f if line.strip())

start_idx = get_processed_count()
print(f"Starting from index {start_idx} / {len(all_pairs)}")

with open(OUTPUT_PATH, "a") as out_f:
    for i in range(start_idx, len(all_pairs)):
        pair = all_pairs[i]

        try:
            description = call_llama(
                pair["pre_obs"],
                pair["action"],
                pair["post_obs"]
            )
            labeled = {
                "pre_obs": pair["pre_obs"],
                "action": pair["action"],
                "state_changes_description": description,
                "post_obs": pair["post_obs"]
            }
            out_f.write(json.dumps(labeled) + "\n")

        except Exception as e:
            print(f"Error on record {i}: {e}")
            labeled = {
                "pre_obs": pair["pre_obs"],
                "action": pair["action"],
                "state_changes_description": "",
                "post_obs": pair["post_obs"]
            }
            out_f.write(json.dumps(labeled) + "\n")

        if (i + 1) % 10 == 0:
            out_f.flush()
            print(f"Processed {i+1}/{len(all_pairs)}")

        if (i + 1) % 500 == 0:
            out_f.flush()
            # auto-save to Drive every 500 records
            os.system('cp /content/worldwideweb/data/raw/wm_training_pairs_labeled.jsonl "/content/drive/MyDrive/World Wide Web/cleaned_data/wm_training_pairs_labeled.jsonl"')
            print(f"--- Checkpoint saved to Drive at {i+1}/{len(all_pairs)} ---")

print("Done!")